# **Interactive exploration of the ArcticNet 1305 CTD route**

*What do temperature, salinity, and other hydrographic properties reveal along the Northwest Passage CTD stations?*

For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/1IzPG-icLlEYTVhEH_7iK6gbD_RlFahI3#scrollTo=E-RxAxv0t-yN)

*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Interactive exploration of the ArcticNet 1305 CTD route](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2FArcticNet_1305_route.ipynb)


**Scientific background**

Hydrographic observations collected across the Canadian Arctic provide essential information on water-mass properties, freshwater influence, stratification, and exchanges between Arctic basins and channels.

CTD measurements support the investigation of temperature and salinity patterns along complex routes such as the Northwest Passage, where spatial variability is influenced by bathymetry, sea ice, river discharge, and connections with the Pacific and Atlantic oceans.

**Cruise overview**

This notebook visualizes the ArcticNet 1305 CTD stations from the Northwest Passage. It converts the observations into an interactive map in which station colours can be updated according to the selected numerical parameter.

**Notebook objectives**

This notebook enables users to:

- retrieve ArcticNet 1305 CTD observations;
- prepare station coordinates and numerical variables;
- retain the shallowest available value at each station;
- explore spatial variability along the Northwest Passage route.

**Data sources**

The notebook accesses https://erddap.emodnet-physics.eu/erddap/tabledap/ARICE_ArcticNet_1305_CTD_NorthwestPassage_v20140708.html through the EMODnet Physics ERDDAP service.

The dataset contains station positions, depths, times, and hydrographic measurements. The processing routine detects the coordinate fields, converts numerical values, removes invalid records, and selects the shallowest observation for each station.

**How to use this notebook**

1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.

**Data retrieval and preparation**

The code cell below performs the complete workflow: library import, ERDDAP retrieval, data cleaning, station-level selection, variable detection, and creation of the interactive map.

 **Interactive visualization**

Choose a numerical parameter from the dropdown menu to update the marker colours.

- The markers identify CTD stations.
- The continuous colour scale represents the selected variable.
- Popups provide local station information.
- The route line provides an approximate view of the sequence of sampled stations.

In [ ]:
# @title
import pandas as pd
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from IPython.display import display
import ipywidgets as widgets
import numpy as np

DATA_URL = 'https://erddap.emodnet-physics.eu/erddap/tabledap/ARICE_ArcticNet_1305_CTD_NorthwestPassage_v20140708.csv'

def load_and_preprocess_data(url):
    """Loads CTD data and keeps only the surface (shallowest-depth) value per station."""
    try:
        df = pd.read_csv(url, skiprows=[1])

        lat_col = [col for col in df.columns if 'latitude' in col.lower()][0]
        lon_col = [col for col in df.columns if 'longitude' in col.lower()][0]
        df = df.rename(columns={lat_col: 'latitude', lon_col: 'longitude'})

        df = df.dropna(subset=['latitude', 'longitude'])

        for col in df.columns:
            if 'time' not in col.lower():
                converted = pd.to_numeric(df[col], errors='coerce')
                if not converted.isna().all():
                    df[col] = converted

        depth_cols = [col for col in df.columns if 'depth' in col.lower()]
        if depth_cols:
            depth_col = depth_cols[0]
            time_col = [col for col in df.columns if 'time' in col.lower()]
            group_cols = ['latitude', 'longitude'] + (time_col[:1] if time_col else [])
            df = df.sort_values(depth_col).groupby(group_cols, as_index=False).first()

        return df
    except Exception as e:
        print(f"Error loading data from {url}: {e}")
        return pd.DataFrame()

df_arcticnet = load_and_preprocess_data(DATA_URL)
print("Data loading complete.")

VARIABLE_METADATA = {
    "depth": ("Depth", "m"),
    "TEMP": ("Sea water temperature", "°C"),
    "TUR3": ("Light transmission", "%"),
    "FLU2": ("Chlorophyll-a fluorescence", "mg/m3"),
    "PSAL": ("Practical salinity", "PSU"),
    "DENS": ("Sea density [sigma-theta]", "kg/m3"),
    "POTENTIAL_TEMP": ("Sea potential temperature", "°C"),
    "SIGMA_THETA": ("Sea sigma-theta", "kg/m3"),
    "DOX1": ("Dissolved oxygen", "ml/l"),
    "NTRA": ("Nitrate [NO3-N]", "MMole/M3"),
    "LGHT": ("Immersed incoming photosynthetic active radiation", "mE/m2/s"),
    "LGH4": ("Surface incoming photosynthetic active radiation", "umole/[m2/s]"),
    "AMON": ("Absolute salinity", "g/kg"),
}

def get_variable_label(variable_name):
    display_name, unit = VARIABLE_METADATA.get(variable_name, (variable_name, ""))
    return f"{display_name} ({unit})" if unit else display_name

def plot_colored_route(variable_name):
    if variable_name not in df_arcticnet.columns:
        print(f"Variable not found: {variable_name}")
        return

    df = df_arcticnet.dropna(subset=["latitude", "longitude", variable_name])
    if df.empty:
        print("No valid data available for this variable.")
        return

    display_name, unit = VARIABLE_METADATA.get(variable_name, (variable_name, ""))
    label = get_variable_label(variable_name)

    m = folium.Map(location=[df["latitude"].mean(), df["longitude"].mean()], zoom_start=5)

    vmin, vmax = df[variable_name].min(), df[variable_name].max()
    colormap = plt.get_cmap("viridis")
    norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin != vmax else colors.Normalize(vmin=vmin - 1, vmax=vmax + 1)

    for _, row in df.iterrows():
        val = row[variable_name]
        hex_color = colors.rgb2hex(colormap(norm(val))[:3])
        popup_text = f"<b>{display_name}</b><br>{val:.2f} {unit}" if unit else f"<b>{display_name}</b><br>{val:.2f}"
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5, color=hex_color, fill=True, fill_color=hex_color,
            fill_opacity=0.8, popup=folium.Popup(popup_text, max_width=300)
        ).add_to(m)

    display(m)

    fig, ax = plt.subplots(figsize=(8, 1))
    fig.subplots_adjust(bottom=0.5)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=colormap), cax=ax, orientation="horizontal", label=label)
    plt.show()

numeric_cols = df_arcticnet.select_dtypes(include=[np.number]).columns.tolist()
exclude_keywords = ["latitude", "longitude", "time", "depth"]
vars_available = [c for c in numeric_cols if not any(x in c.lower() for x in exclude_keywords) and not c.upper().endswith("_QC")]

variable_dropdown = widgets.Dropdown(
    options=[(get_variable_label(c), c) for c in vars_available],
    description="Variable:"
)
output_map = widgets.Output()

def on_change(change):
    with output_map:
        output_map.clear_output(wait=True)
        if variable_dropdown.value:
            plot_colored_route(variable_dropdown.value)

variable_dropdown.observe(on_change, "value")
display(variable_dropdown, output_map)
on_change(None)

Data loading complete.


Dropdown(description='Variable:', options=(('Sea water temperature (°C)', 'TEMP'), ('Light transmission (%)', …

Output()

**Interpretation guidance**

The visualization represents surface or near-surface conditions because the shallowest valid record is retained for each station. It does not reproduce complete CTD profiles.

Differences between stations may reflect real environmental gradients, but also changes in sampling time, local bathymetry, station depth, or data coverage. The route line is schematic and should not be treated as the exact ship track.


**Additional resources**

Libraries used:
*  [pandas](https://pandas.pydata.org/) for data handling
*  numpy for data handling
*  folium for interactive mappingg
*  [matplotlib](https://matplotlib.org/) for colour normalization
*  [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/) for the parameter selector.


This work has received funding from the European Union Horizon Europe project Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN ICE) under grant agreement No. 101060452 (https://doi.org/10.3030/101060452). UK partners are funded by UK Research and Innovation (UKRI) under the UK government's Horizon Europe funding guarantee.

This notebook makes use of data from the ARICE (Arctic Research Icebreaker Consortium) project, hosted via EMODnet Physics (https://www.emodnet-physics.eu).

<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 20px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
    <img src="https://emodnet.ec.europa.eu/sites/emodnet.ec.europa.eu/files/public/emodnet_logos/web/EMODnet_standard_colour.png" height="100"/>
  </div>
</center>